[문항 1] 주관식 PyTorch의 ImageFolder를 사용하여 이미지 데이터를 로드하려고 합니다. 이때 데이터셋 디렉터리(root/) 내부는 어떤 구조로 정리되어 있어야 합니까? 하위 폴더의 이름은 무엇을 의미하게 되는지 서술하시오.

train과 val 두 폴더로 나뉘어져 있으며, 그 안에는 각 클래스 이름의 하위 폴더가 있다.

[문항 2] 주관식 강의에서 다룬 '시베리안 허스키 vs 늑대' 분류 사례처럼 훈련 데이터가 매우 적은 경우(예: 40장), 모델의 성능을 높이고 과적합(Overfitting)을 방지하기 위해 적용할 수 있는 주요 기법 두 가지를 설명하시오. (힌트: 모델 학습 방식과 데이터 처리 방식 측면에서 각각 하나씩)

전이 학습으로 사전 학습 모델의 가중치를 동결하고, 마지막 분류 계층만 새로 학습한다. RandomHorizontalFlip, RandomErasing과 같은 데이터 증강을 진행한다.

[문항 3] 코드 구현 검증(Validation) 데이터셋을 위한 전처리 코드를 작성하시오.

요구사항:

이미지 크기를 256으로 변경 (Resize)

중앙을 224x224 크기로 자름 (CenterCrop)

텐서(Tensor) 형태로 변환

정규화 (평균: 0.5, 표준편차: 0.5) 적용

In [ ]:
import torchvision.transforms as transforms

# 검증 데이터용 transform 정의
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

## Resize: 가로 세로 비율 유지하면서 크기 변화
# 결과가 256*256이 아님, 짧은 변을 256으로 하고 비율 맞춰서 다른 변 정함
# 왜 256임?
# 나중에 이미지를 224 크기로 자를 때 여유 있는 마진을 두기 위해

## CenterCrop
# 이미지 중앙에 위치시키고 224*224로 자르기 (정보 손실)
# 하는 이유?
# Resize만 하면 정사각형 이미지를 만들지 않음. 원본 비율에 맞게 크기 조정하기 때문
# 정사각형 이미지를 만들기 위해, 가운데만 자르는 것

## 224*224로 맞춰주는 이유
# ImageNet 대회 모델들이 224*224 를 입력으로 받음
# AlexNet, VGGNet같은 초기 모델들이 GPU 메모리, 연산량 때문에 최적화해둔 크기임
# (이미지 너무 크면 느리고 작으면 성능이 떨어기 때문에 적당한 사이즈로 정해둔 것)

[문항 4] 코드 구현 훈련(Train) 데이터셋을 위한 전처리 코드를 작성하시오. 데이터 부족 문제를 완화하기 위해 데이터 증강(Data Augmentation) 기법을 포함해야 합니다.

요구사항:

무작위 크기 및 비율로 자른 후 224로 크기 변경 (RandomResizedCrop)

무작위로 수평 뒤집기 (RandomHorizontalFlip)

텐서(Tensor) 형태로 변환

정규화 (평균: 0.5, 표준편차: 0.5) 적용

무작위로 영역 지우기 (RandomErasing) - (확률 0.5, scale 등은 강의 코드 참조 혹은 기본값)

In [ ]:
# 훈련 데이터용 transform 정의 (데이터 증강 적용)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.33),
                             ratio = (0.3, 3.3), value = 0, inplace = False),
    transforms.Normalize((0.5, 0.5, 0,5), (0.5, 0.5, 0.5))
])

## RandomErasing
# p: 적용 확률
# scale: 지울 면적 범위(현재 2~33% 사이 랜덤한 면적을 지움)
# ratio: 지울 사각형의 비율
# value: 지운 영역을 채울 픽셀값
# inplace: 입력 텐서 수정 여부 (True면 입력 텐서 자체가 바뀌고, False면 사본에 Erase 적용 후 반환)

## 순서 주의할 것
# RandomErasing은 Tensor 입력만 지원함 -> ToTensor 이후에 실행해야 함
# Normalize는 항상 마지막에 두는게 일반적

[문항 5] 코드 구현 위에서 정의한 transform을 사용하여 ImageFolder로 데이터셋을 불러오는 코드를 작성하시오.

train_dir: 훈련 데이터 경로

test_dir: 검증 데이터 경로

In [ ]:
import torchvision.datasets as datasets
import os

# 경로 설정 (예시 경로)
data_dir = 'hymenoptera_data'
train_dir = os.path.join(data_dir, 'train')
test_dir = os.path.join(data_dir, 'val')

# ImageFolder를 이용한 데이터셋 정의
# ( 여기에 코드를 작성하시오 )
train_data = datasets.ImageFolder(train_dir, transform=train_transform)
test_data = datasets.ImageFolder(test_dir, transform=test_transform)

[문항 6] 코드 구현 불러온 데이터셋을 모델에 공급하기 위한 DataLoader를 정의하시오.

배치 사이즈(batch_size): 5

훈련용 로더: 데이터를 섞어서(shuffle=True) 로드

검증용 로더: 데이터를 섞지 않고(shuffle=False) 로드

In [ ]:
from torch.utils.data import DataLoader

batch_size = 5

# 데이터 로더 정의
# ( 여기에 코드를 작성하시오 )
train_loader = DataLoader(train_data,
                          batch_size=batch_size,
                          shuffle=True)
test_loader = DataLoader(test_data,
                         batch_size=batch_size,
                         shuffle=False)

In [ ]:
## torchvision.datasets.ImageFolder와 torch.utils.data.DataLoader를 헷갈리지 말 것

## 이전까지는 ImageFolder 안 쓴 이유?
# ImageFolder는 '사용자 로컬 폴더'를 읽을 때 씀
# 폴더 안에 있는 이미지마다 라벨(폴더명)을 매핑해줌
# 이를 위해서는 폴더의 형식이 정해져 있어야 함

# data/                 # 데이터는 train, val 폴더로 구성되어야 함
#  ├─ train/          # train, val 하위 폴더는 각 클래스로 나뉘어 있어야 함
#  │   ├─ ants/      # class 0
#  │   └─ bees/      # class 1
#  └─ val/
#      ├─ ants/
#      └─ bees/

# DataLoader는 배치로 묶기, 묶는 과정에서 섞기(shuffle 설정에 따라)

## CIFAR10 쓸 때는 ImageFolder 필요 없는 이유?
# torchvision 내장 데이터셋이라서 (이미지, 라벨)을 바로 제공해줌
# 따로 데이터-라벨 매핑해줄 필요가 없는 것

# CIFAR10 데이터셋 불러오는 예시
trainset = datasets.CIFAR10(    # ImageFolder 없이 바로 불러옴
            root = './data',
            train = True,
            download = True,
            transform = transform   # transform도 데이터 불러올 때 적용
)

trainloader = DataLoader(    # 데이터 가져온 후에 배치로 묶기
              trainset,
              batch_size = batch_size,
              shuffle = True
)

자율 학습